# GeoReady 01 · Eligible land and candidate zones

1. Load the prepared 250 m grids.
2. Give every cell an exclusion reason. **Protected areas are a legal prohibition** and always win;
   land cover, slope, weak resource and distance to a substation are **editable technical
   thresholds**, reported separately so none is silently treated as law. A missing input is
   coded *unknown*, never *eligible*.
3. Cut eligible land into 10 km squares that never cross a GCCA substation zone, and keep
   pieces of at least 25 km² as candidate zones.

In [ ]:
import os, sys
from pathlib import Path
def _root():  # the geoready folder: from GEOREADY_ROOT, the working folder, or the QGIS Notebook plugin
    if os.environ.get("GEOREADY_ROOT"):
        return Path(os.environ["GEOREADY_ROOT"])
    starts = [Path.cwd()]
    try:
        from qgis.PyQt.QtCore import QSettings
        starts.append(Path(QSettings().value("QGISNotebook/last_directory", "") or "."))
    except ImportError:
        pass
    return next((p for s in starts for p in [s, *s.parents] if (p / "src" / "geoready").exists()), None)
ROOT = _root()
assert ROOT, "Open this notebook from inside the geoready folder, or set GEOREADY_ROOT"
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
if "qgis" in sys.modules:          # inside QGIS: draw figures off-screen
    import matplotlib; matplotlib.use("Agg")
from geoready.nb import show
CONFIG = "config/northern_cape_pv.yaml"

In [ ]:
import numpy as np, pandas as pd
from geoready.run import Run
from geoready import zones, plots
run = Run(CONFIG); cfg = run.cfg
g, profile = zones.load_grids(cfg)
# Legal exclusion from the polygons, every touched cell
g["protected"] = zones.rasterize_mask(cfg["vectors"], cfg["protected_layer"], profile).astype("float32")
cell_km2 = (profile["transform"].a / 1000) ** 2
code = zones.eligibility(g, cfg)

In [ ]:
summary = pd.DataFrame({"reason": zones.CODES}).assign(
    km2=[float((code == c).sum() * cell_km2) for c in zones.CODES])
summary["share"] = summary["km2"] / summary["km2"].sum()
show(summary.round({"km2": 0, "share": 3}))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 7))
plots.eligibility_map(code, profile, zones.CODES, ax=ax)
show(fig, save_as=run.path("eligibility_map.png"))

In [ ]:
lab = zones.zone_ids(code == 0, g["gcca_zone"], profile["transform"], cfg["zones"]["size_km"])
table, zone_grid = zones.zone_table(lab, g, cell_km2, cfg["zones"]["min_eligible_km2"])
print(len(table), "zones,", round(table["eligible_km2"].sum()), "km² of eligible land in zones")
show(table.describe().T[["min", "50%", "max"]].round(2))

Save the exclusion grid, the zone grid and the zone table for the next notebooks.

In [ ]:
import rasterio
p = profile.copy(); p.update(dtype="float32", nodata=-9999, compress="deflate")
with rasterio.open(run.path("eligibility.tif"), "w", **p) as ds:
    ds.write(np.where(np.isnan(code), -9999, code).astype("float32"), 1)
p.update(dtype="int32", nodata=0)
with rasterio.open(run.path("zone_id.tif"), "w", **p) as ds:
    ds.write(zone_grid.astype("int32"), 1)
p.update(dtype="uint8", nodata=None)
with rasterio.open(run.path("protected_mask.tif"), "w", **p) as ds:
    ds.write(g["protected"].astype("uint8"), 1)
fp = zones.footprints(zone_grid, profile["transform"], profile["crs"], list(table["zone_id"]))
fp.to_file(run.path("zones_footprint.gpkg"), layer="zones", driver="GPKG")
table.to_csv(run.path("zones_stage1.csv"), index=False)
summary.to_csv(run.path("eligibility_summary.csv"), index=False)